# GIU portal basics, one small step at a time

Same gentle idea as the CMS notebook. No agent, nothing clever. One cell, one
small thing. Press **Shift + Enter** and read the result.

This notebook is tailored to the **GIU** student portal. The confirmed main
workflow is the transcript: choose an academic year, load its courses, then
search those courses and letter grades locally.

Note: the portal is old and slow. A cell may take up to a minute to answer. That
is normal here. Just wait for it.

## Step 1 — log in

In [17]:
import os, getpass
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")  # reads the portal package's local .env
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

# Reload local package modules so rerunning this cell picks up source fixes
# even when the Jupyter kernel imported an older version earlier.
import importlib
import guc_portal._sites as portal_sites_module
import guc_portal.client as portal_client_module

importlib.reload(portal_sites_module)
importlib.reload(portal_client_module)
GucPortal = portal_client_module.GucPortal

portal = GucPortal(site="giu")
print("Portal client configured.")
print("Site:", portal.site_name)
print("Transcript route:", portal.site.transcript_path)

Portal client configured.
Site: giu
Transcript route: /EXTStudent/Transcript_m.aspx


### GIU profile

Step 1 explicitly selects GIU. The client first visits GIU's Home page to establish
the session, then uses the confirmed transcript URL. The current-grade and previous
detailed-grade pages are optional: they can legitimately return no courses or terms.

## Step 2 — transcript years

`available_years()` reads the academic-year dropdown from GIU's confirmed
`Transcript_m.aspx` page.

In [18]:
try:
    years = portal.available_years()
    print("Your transcript years:\n")
    for code, name in years:
        print(f" - {name} (portal code: {code})")
except Exception as exc:
    print(type(exc).__name__, exc)
    print("The portal may be busy. Wait about one minute, then run Step 2 again.")

Your transcript years:

 - 2003-2004 (portal code: 1)
 - 2004-2005 (portal code: 2)
 - 2005-2006 (portal code: 3)
 - 2006-2007 (portal code: 4)
 - 2007-2008 (portal code: 5)
 - 2008-2009 (portal code: 6)
 - 2009-2010 (portal code: 7)
 - 2010-2011 (portal code: 8)
 - 2011-2012 (portal code: 9)
 - 2012-2013 (portal code: 10)
 - 2013-2014 (portal code: 11)
 - 2014-2015 (portal code: 12)
 - 2015-2016 (portal code: 13)
 - 2016-2017 (portal code: 14)
 - 2017-2018 (portal code: 15)
 - 2018-2019 (portal code: 16)
 - 2019-2020 (portal code: 17)
 - 2020-2021 (portal code: 18)
 - 2021-2022 (portal code: 19)
 - 2022-2023 (portal code: 20)
 - 2023-2024 (portal code: 21)
 - 2024-2025 (portal code: 22)
 - 2025-2026 (portal code: 23)
 - 2026-2027 (portal code: 24)
 - 2027-2028 (portal code: 25)


## Step 3 — transcript for one year

Choose a year by its visible label. The client submits the same dropdown form that
the browser uses and returns the GPA plus every course and letter grade. If you just
ran Step 2, the portal may make this cell wait about a minute before retrying.

In [19]:
year_name = "2024-2025"  # change this to any label printed in Step 2
try:
    if "years" not in globals():
        years = portal.available_years()
    year_code = next((value for value, label in years if year_name in label), None)
    if year_code is None:
        print("No transcript year named", year_name, "— choose one from Step 2.")
    else:
        transcript = portal.get_transcript_year(year_code, tries=2, delay=60)
        print("Cumulative GPA:", transcript.cumulative_gpa, "\n")
        for row in transcript.rows:
            print(f"{row.semester}: {row.grade} — {row.course}")
except Exception as exc:
    print(type(exc).__name__, exc)
    print("The portal may be busy. Wait about one minute, then run Step 3 again.")

Cumulative GPA: 2.68 

Winter 2024: B- — Operating Systems
Winter 2024: C- — Software Project I
Winter 2024: C- — Software and Mobile Devices Security
Winter 2024: B — Requirements Engineering
Winter 2024: A- — Software Design and Architecture
Winter 2024: A- — Software Construction and Testing
Winter 2024: B+ — Data Engineering and Visualization
Spring 2025: A — Big Data and NoSQL
Spring 2025: B+ — Software Cloud Computing
Spring 2025: B+ — Software Mobile Development
Spring 2025: A+ — Software Project II
Spring 2025: B- — Project management
Spring 2025: A+ — Research Methodology
Spring 2025: B — Mobile Development
Summer 2025 Round 2: A+ — Workplace & Internship Readiness


## Step 4 — find a course grade by name

This searches the transcript already loaded in Step 3, so it makes **no new portal
request**. Enter part of a course title or code.

In [20]:
if "transcript" not in globals():
    raise RuntimeError("Run Step 3 successfully before searching its courses.")

# Edit this value to search for another course title or course code.
course_query = "Software Project II"
needle = course_query.strip().lower()
if not needle:
    raise ValueError("course_query cannot be empty.")

matches = [
    row for row in transcript.rows
    if needle in (row.course or "").lower()
]
if matches:
    for row in matches:
        print(f"{row.course}: {row.grade} ({row.semester}, {row.hours} hours)")
else:
    print(f"No course in {year_name} matches {course_query!r}.")
    print("Available courses:")
    for row in transcript.rows:
        print(" -", row.course)

Software Project II: A+ (Spring 2025, 4 hours)


## Step 5 — current courses (optional)

`list_current_courses()` checks GIU's current-grade page. It can be empty between
terms. Wait about one minute after the last portal request before running it.

In [21]:
try:
    current_courses = portal.list_current_courses()
    if current_courses:
        print("Current courses:\n")
        for code, name in current_courses:
            print(f" - {name} (portal code: {code})")
    else:
        print("No current courses were returned; this is normal between terms.")
except Exception as exc:
    print(type(exc).__name__, exc)
    print("The portal may be busy. Wait about one minute, then run Step 5 again.")

No current courses were returned; this is normal between terms.


## Step 6 — list detailed-grade seasons

First read the **Season** dropdown on GIU's previous-grades page. Each item has
a readable label, such as `Winter 2025`, and an internal value used by the portal.

In [22]:
detailed_seasons = []
try:
    detailed_seasons = portal.available_seasons(tries=2, delay=60)
    if detailed_seasons:
        print("Available detailed-grade seasons:\n")
        for value, label in detailed_seasons:
            print(f" - {label} (portal code: {value})")
    else:
        print("No detailed-grade seasons were returned.")
except Exception as exc:
    print(type(exc).__name__, exc)
    print("The portal may be busy. Wait about one minute, then run Step 6 again.")

Available detailed-grade seasons:

 - Spring 2027 (portal code: 71)
 - Winter 2026 (portal code: 70)
 - Summer 2026 Round 1 (portal code: 69)
 - Spring 2026 (portal code: 68)
 - Winter 2025 (portal code: 67)
 - Summer 2025 Round 1 (portal code: 66)
 - Spring 2025 (portal code: 65)
 - Winter 2024 (portal code: 64)
 - Summer 2024 Round 1 (portal code: 63)
 - Spring 2024 (portal code: 62)
 - Winter 2023 (portal code: 61)
 - Summer 2023 Round 1 (portal code: 60)
 - Spring 2023 (portal code: 59)
 - Winter 2022 (portal code: 58)
 - Summer 2022 Round 1 (portal code: 57)
 - Spring 2022 (portal code: 56)
 - Winter 2021 (portal code: 55)
 - Summer 2021 Round 1 (portal code: 54)
 - Spring 2021 (portal code: 53)
 - Winter 2020 (portal code: 52)
 - Summer 2020 Round 1 (portal code: 51)
 - Spring 2020 (portal code: 50)


## Step 7 — choose a season and list its courses

Selecting a season makes GIU populate the **Course** dropdown. Change
`season_name` to one of the labels printed in Step 6. This cell then prints the
course labels exactly as GIU writes them.

In [25]:
season_name = "Winter 2024"  # copy a season label from Step 6
detailed_courses = []
selected_season = None
season_options = globals().get("detailed_seasons", [])

season_matches = [
    (value, label)
    for value, label in season_options
    if season_name.casefold() in label.casefold()
]

if not season_matches:
    print(f"No season matches {season_name!r}. Run Step 6 and copy one of its labels.")
elif len(season_matches) > 1:
    print("That season text matches more than one option. Be more specific:")
    for _value, label in season_matches:
        print(" -", label)
else:
    season_value, selected_season = season_matches[0]
    try:
        detailed_courses = portal.list_previous_courses(
            season_value,
            tries=2,
            delay=60,
        )
        print(f"Courses in {selected_season}:\n")
        for value, label in detailed_courses:
            print(f" - {label} (portal code: {value})")
        if not detailed_courses:
            print(" - No courses were returned for this season.")
    except Exception as exc:
        print(type(exc).__name__, exc)
        print("The portal may be busy. Wait about one minute, then run Step 7 again.")

Courses in Winter 2024:

 - GIU-Cairo.Informatics and Computer Science 3rd - INCS102 Operating Systems (portal code: 4602)
 - GIU-Cairo.Informatics and Computer Science - Software Engineering 5th - ICS501 Software Project I (portal code: 4724)
 - GIU-Cairo.Informatics and Computer Science - IT Security 5th - ICS506 Software and Mobile Devices Security (portal code: 4730)
 - GIU-Cairo.Informatics and Computer Science - Software Engineering 5th - ICS508 Requirements Engineering (portal code: 4732)
 - GIU-Cairo.Informatics and Computer Science - Software Engineering 5th - ICS509 Software Design and Architecture (portal code: 4733)
 - GIU-Cairo.Informatics and Computer Science - Software Engineering 5th - ICS510 Software Construction and Testing (portal code: 4734)
 - GIU-Cairo.Informatics and Computer Science - Data Science 5th - ICS503 Data Engineering and Visualization (portal code: 4845)


## Step 8 — choose a course and fetch its grades

GIU writes a course as one long label, for example:

`GIU-Cairo. Informatic and Computer Science Thesis & Internship - INCS700 Bachelor Thesis`

You may paste the full label, but a unique fragment such as `INCS700` is easier
and safer. The cell selects the matching portal value, then fetches the assessment
rows and the midterm-results table.

In [24]:
course_query = "INCS700"  # full GIU label or a unique fragment/course code
course_options = globals().get("detailed_courses", [])

course_matches = [
    (value, label)
    for value, label in course_options
    if course_query.casefold() in label.casefold()
]

if not course_matches:
    print(f"No course matches {course_query!r}. Run Step 7 and copy a label or code.")
elif len(course_matches) > 1:
    print("That course text matches more than one option. Use its unique code or full label:")
    for _value, label in course_matches:
        print(" -", label)
else:
    course_value, selected_course = course_matches[0]
    try:
        grades = portal.get_previous_grades(
            season_value,
            course_value,
            tries=2,
            delay=60,
        )

        print(f"Selected: {selected_course}")
        print(f"Season: {selected_season}")
        print("\nAssessment grades:")
        if grades.items:
            for item in grades.items:
                print(f" - {item.assessment} / {item.element}: {item.grade} — {item.evaluator}")
        else:
            print(" - No assessment rows were displayed for this course.")

        print("\nMidterm results:")
        if grades.percentages:
            for course, percentage in grades.percentages.items():
                print(f" - {course}: {percentage}")
        else:
            print(" - No midterm percentages were displayed.")
    except Exception as exc:
        print(type(exc).__name__, exc)
        print("The portal may be busy. Wait about one minute, then run Step 8 again.")

Selected: GIU-Cairo.Informatic and Computer Science Thesis & Internship - INCS700 Bachelor Thesis
Season: Winter 2025

Assessment grades:
 - Quiz/Assignment Element Name Grade Prof./Lecturer/TA Project Question1 40
															/
															40 Nada Ahmed Hamed Sharaf Question2 15
															/
															15 Nada Ahmed Hamed Sharaf Question3 13
															/
															15 Nada Ahmed Hamed Sharaf Question4 14
															/
															15 Nada Ahmed Hamed Sharaf Question5 14
															/
															15 Nada Ahmed Hamed Sharaf / Quiz/Assignment: Element Name — Grade
 - Quiz/Assignment / Element Name: Grade — Prof./Lecturer/TA
 - Project / Question1: 40 / 40 — Nada Ahmed Hamed Sharaf
 -  / Question2: 15 / 15 — Nada Ahmed Hamed Sharaf
 -  / Question3: 13 / 15 — Nada Ahmed Hamed Sharaf
 -  / Question4: 14 / 15 — Nada Ahmed Hamed Sharaf
 -  / Question5: 14 / 15 — Nada Ahmed Hamed Sharaf
 - Project / Question1: 40 / 40 — Nada Ahmed Hamed Sharaf
 -  / Question2: 

## That is all

You used the GIU-supported path first:

- `available_years()` — transcript dropdown years
- `get_transcript_year(code)` — courses, letter grades, and GPA for one year
- local name search — find a transcript course without another request
- `list_current_courses()` — optional current-term availability
- `available_seasons()` — labels from GIU's detailed-grade season dropdown
- `list_previous_courses(code)` — exact course labels inside one season
- `get_previous_grades(season, course)` — assessments and midterm results

No agent, no magic. Just a few functions that read the portal for you. If a cell
gives an error, the portal was busy: wait a minute and run it again.